# 🛒 Visual Product Search — Final Compare
FINAL COMPARE: Merge A+B and C results into a single table.
Requires:
- eval_ab_results.json (from eval_ab notebook)
- eval_c_results.json (from eval_c notebook)

Upload both as Kaggle datasets, then run this notebook.


In [1]:
# ── Install ──
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas'])

import os, json, numpy as np, pandas as pd


In [2]:
# ── Config ──
# ⚠️ UPDATE THESE paths to match your input datasets
AB_DIR = '/kaggle/input/notebooks/chandrahas7c/eval-a-b'
C_DIR = '/kaggle/input/notebooks/chandrahas7c/eval-c'
WORK = '/kaggle/working'
os.makedirs(WORK, exist_ok=True)

K_VALUES = [5, 10, 15]


In [3]:
# ── Load results ──
ab_path = f'{AB_DIR}/eval_ab_results.json'
c_path = f'{C_DIR}/eval_c_results.json'

# Try alternative paths if not found
for p in [ab_path, f'{WORK}/eval_ab_results.json']:
    if os.path.exists(p): ab_path = p; break
for p in [c_path, f'{WORK}/eval_c_results.json']:
    if os.path.exists(p): c_path = p; break

print(f'Loading A+B results from: {ab_path}')
with open(ab_path) as f: ab_results = json.load(f)

print(f'Loading C results from: {c_path}')
with open(c_path) as f: c_results = json.load(f)


Loading A+B results from: /kaggle/input/notebooks/chandrahas7c/eval-a-b/eval_ab_results.json
Loading C results from: /kaggle/input/notebooks/chandrahas7c/eval-c/eval_c_results.json


In [4]:
# ── Merge ──
seed_results = {**ab_results, **c_results}
print(f'Merged configs: {list(seed_results.keys())}')


Merged configs: ['A (α=1.0)', 'B (α=0.3)', 'B (α=0.7)', 'C (α=0.3)', 'C (α=0.7)']


## ══════════ FINAL RESULTS TABLE ══════════


In [5]:
# ══════════ FINAL RESULTS TABLE ══════════
config_order = ['A (α=1.0)', 'B (α=0.3)', 'B (α=0.7)', 'C (α=0.3)', 'C (α=0.7)']
rows = []
for name in config_order:
    if name not in seed_results:
        print(f'⚠️ Missing config: {name}')
        continue
    res = seed_results[name]
    for k in K_VALUES:
        sk = str(k)
        rows.append({
            'Config': name, 'K': k,
            'Hit@K': f"{np.mean(res[sk]['hit']):.4f} ± {np.std(res[sk]['hit']):.4f}",
            'Recall@K': f"{np.mean(res[sk]['recall']):.4f} ± {np.std(res[sk]['recall']):.4f}",
            'NDCG@K': f"{np.mean(res[sk]['ndcg']):.4f} ± {np.std(res[sk]['ndcg']):.4f}",
            'mAP@K': f"{np.mean(res[sk]['map']):.4f} ± {np.std(res[sk]['map']):.4f}",
        })

results_df = pd.DataFrame(rows)
results_df.to_csv(f'{WORK}/final_results.csv', index=False)

print('\n'+'='*75)
print(f'FINAL RESULTS  |  Configs: {len(config_order)}')
print('='*75)
print(results_df.to_string(index=False))
print(f'\n✅ Saved to {WORK}/final_results.csv')



FINAL RESULTS  |  Configs: 5
   Config  K           Hit@K        Recall@K          NDCG@K           mAP@K
A (α=1.0)  5 0.4544 ± 0.0000 0.1689 ± 0.0000 0.2124 ± 0.0000 0.1328 ± 0.0000
A (α=1.0) 10 0.5214 ± 0.0000 0.2077 ± 0.0000 0.2154 ± 0.0000 0.1423 ± 0.0000
A (α=1.0) 15 0.5577 ± 0.0000 0.2314 ± 0.0000 0.2212 ± 0.0000 0.1460 ± 0.0000
B (α=0.3)  5 0.4193 ± 0.0000 0.1731 ± 0.0000 0.1904 ± 0.0000 0.1242 ± 0.0000
B (α=0.3) 10 0.5020 ± 0.0000 0.2189 ± 0.0000 0.2007 ± 0.0000 0.1345 ± 0.0000
B (α=0.3) 15 0.5263 ± 0.0000 0.2357 ± 0.0000 0.2046 ± 0.0000 0.1372 ± 0.0000
B (α=0.7)  5 0.4866 ± 0.0000 0.1961 ± 0.0000 0.2229 ± 0.0000 0.1437 ± 0.0000
B (α=0.7) 10 0.5598 ± 0.0000 0.2383 ± 0.0000 0.2304 ± 0.0000 0.1543 ± 0.0000
B (α=0.7) 15 0.5880 ± 0.0000 0.2571 ± 0.0000 0.2353 ± 0.0000 0.1577 ± 0.0000
C (α=0.3)  5 0.6092 ± 0.0011 0.2842 ± 0.0008 0.3035 ± 0.0006 0.2060 ± 0.0005
C (α=0.3) 10 0.7093 ± 0.0017 0.3736 ± 0.0012 0.3288 ± 0.0007 0.2296 ± 0.0006
C (α=0.3) 15 0.7454 ± 0.0018 0.4198 ± 0.0013 0

In [6]:
# ── Also save merged JSON ──
with open(f'{WORK}/all_results_merged.json', 'w') as f:
    json.dump(seed_results, f, indent=2)
print(f'✅ Merged JSON saved to {WORK}/all_results_merged.json')

print('\n✅✅✅ FINAL COMPARISON COMPLETE ✅✅✅')


✅ Merged JSON saved to /kaggle/working/all_results_merged.json

✅✅✅ FINAL COMPARISON COMPLETE ✅✅✅
